## Download collection and create index

Based on [this](https://github.com/castorini/pyserini/blob/master/docs/experiments-msmarco-v2.md) guide.
This is a lengthy process, as there are 138,364,198 documents total.

```bash
# In the root folder
wget -P /collections --header "X-Ms-Version: 2019-12-12" https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco_v2_passage.tar
tar -xvf collections/msmarco_v2_passage.tar -C ./collections

python -m pyserini.index.lucene \
  --collection MsMarcoV2PassageCollection \
  --input collections/msmarco_v2_passage \
  --index indexes/lucene-index.msmarco-v2-passage \
  --generator DefaultLuceneDocumentGenerator \
  --threads 12

# Total index size
du -sh indexes/lucene-index.msmarco-v2-passage
du -ah indexes/lucene-index.msmarco-v2-passage | sort -h | tail -20

# Don't forget to remove what's inside collections to free some storage.
rm -rf collections/msmarco_v2_passage
# Remove the index if needed
rm -rf indexes/lucene-index.msmarco-v2-passage

```
Adjust -threads as appropriate. Different configurations (-storePositions, -storeDocvectors, -storeRaw) support different features, but require different amounts of disk space; for the detailed tradeoffs, see the (Anserini)[https://github.com/castorini/anserini/blob/master/docs/experiments-msmarco-v2.md] guide. The above minimal index should be ~11 GB.


## Performing runs

The documentation suggests running the following command:
```bash
python -m pyserini.search.lucene \
  --index indexes/lucene-index.msmarco-v2-passage \
  --topics msmarco-v2-passage-dev \
  --output runs/run.msmarco-v2-passage.dev.txt \
  --batch-size 36 --threads 12 \
  --hits 1000 \
  --bm25
```
But this error occurs: `jnius.JavaException: JVM exception occurred: java.io.IOException: Error downloading topics from https://raw.githubusercontent.com/castorini/anserini-tools/master/topics-and-qrels/topics.msmarco-v2-passage.dev.txt`. The url returns 404, I researched a bit and found that the repository had its [name and folder structure changed](https://github.com/castorini/eval/pull/127/changes). So, while Pyserini doesn't fix this issue I'm using a workaround, downloading all `topics` and `qrels` I need from the [repository](https://github.com/castorini/eval), putting them into folders and passing these files to the command.

```bash
python -m pyserini.search.lucene \
  --index indexes/lucene-index.msmarco-v2-passage \
  --topics topics/topics.msmarco-v2-passage.dev.txt \
  --output runs/run.msmarco-v2-passage.dev.txt \
  --batch-size 36 \
  --threads 12 \
  --hits 1000 \
  --bm25
```


## Generating metrics

```bash
python -m pyserini.eval.trec_eval \
  -c \
  -m map_cut.100 \
  -m recip_rank \
  -m recall.100,1000 \
  -m ndcg_cut.10 \
  -m P.10,100 \
  qrels/qrels.msmarco-v2-passage.dev.txt \
  runs/run.msmarco-v2-passage.dev.txt
```


## BM25

In [33]:
from pyserini.search.lucene import LuceneSearcher

lucene_bm25_searcher = LuceneSearcher('indexes/lucene-index.msmarco-v2-passage')
hits = lucene_bm25_searcher.search('what is a lobster roll?', k=10)

for i in range(len(hits)):
    print(f'{i+1:2} {hits[i].docid:7} {hits[i].score:.5f}')

 1 msmarco_passage_31_146266807 12.99720
 2 msmarco_passage_28_574935093 12.55690
 3 msmarco_passage_46_548883494 12.49980
 4 msmarco_passage_22_790821258 12.25090
 5 msmarco_passage_46_548881650 11.95540
 6 msmarco_passage_64_862014506 11.88230
 7 msmarco_passage_28_575298797 11.85110
 8 msmarco_passage_31_146267697 11.84610
 9 msmarco_passage_53_5702587 11.81510
10 msmarco_passage_63_407817293 11.68570


In [34]:
# The document's raw content
print(hits[0].lucene_document.get('raw'))
# The docid from the collection, type string.
print(hits[0].docid)
# Lucene's internal docid, type int
print(hits[0].lucene_docid)
# Score, type float
print(hits[0].score)
# Raw Lucene document, type org.apache.lucene.document.Document
print(hits[0].lucene_document)

{
  "pid" : "msmarco_passage_31_146266807",
  "passage" : "Previous. Next. The stars of the New England Lobster Roll show at Mason's Famous Lobster Rolls are the two at the top of the menu: The Classic Lobster Roll (fresh lobster with a bit of mayo and lemon on the toasted bun) and what they call the Connecticut Roll (warm, buttered lobster on the regulation bun. Just lobster, nothing else).",
  "spans" : "(3173,3181),(3182,3186),(3187,3477),(3478,3506)",
  "docid" : "msmarco_doc_27_500998168"
}
msmarco_passage_31_146266807
68157426
12.997200012207031
<org.apache.lucene.document.Document at 0x7f632e3d3160 jclass=org/apache/lucene/document/Document jself=<LocalRef obj=0x365d37aa at 0x7f632e344f10>>
